# 01 요일 평균 베이스라인 WAPE (배달 · 홀 분리)

**결론**: 샘플 데이터에서 요일 평균의 WAPE는 배달 0.179 · 홀 0.134, 두 채널을 더한 메뉴별 판매로는 0.115, Bias는 0에 가깝다(+0.007). 샘플에 넣은 노이즈 수준이라 채널을 나눠도 식이 제대로 동작한다는 확인이다. 하루 1~3개 팔리는 조합(돈까스정식 · 생선까스 배달)은 WAPE가 0.3~0.75로 높지만 판매량 가중이라 전체에는 영향이 작다. 영업일 미판매를 0으로 채우지 않으면 돈까스정식 배달 예측 평균이 실제(1.58)보다 26% 큰 2.00으로 부풀려진다(채우면 1.62). 위글리 배달 데이터는 아직 없어 받으면 `DATA_DIR`만 바꿔 다시 돌린다.

- 예측식: `src/weglee_ai/baseline.py` (3주 계획 4.3절). 배달 · 홀을 따로 예측한다 (설계 5.2절)
- 행이 없는 날: 채널 전체 판매 0 = 휴무, 영업일에 메뉴 행 없음 = 0개 판매 (판매 데이터 사전 5절)
- 평가: 날짜마다 그날 이전 데이터만으로 예측 (rolling origin). 지표 식은 `src/weglee_ai/metrics.py`
- 먼저 `uv run python scripts/make_sample_data.py`로 샘플 데이터를 만든다. 실데이터는 `DATA_DIR`만 바꾼다


In [ ]:
from pathlib import Path

import pandas as pd

from weglee_ai.baseline import rolling_forecast
from weglee_ai.metrics import bias, wape

DATA_DIR = Path("..") / "data" / "sample"  # 실데이터는 ../data/processed
EVAL_WEEKS = 4  # 마지막 4주를 평가


def score(g: pd.DataFrame) -> pd.Series:
    return pd.Series({"WAPE": wape(g.actual, g.predicted), "Bias": bias(g.actual, g.predicted), "판매량": g.actual.sum()})


In [ ]:
sales_path = DATA_DIR / "pos_sales.csv"
if not sales_path.exists():
    raise FileNotFoundError(f"{sales_path} 없음 — ai/ 에서 `uv run python scripts/make_sample_data.py` 먼저 실행")

sales = pd.read_csv(sales_path, parse_dates=["sold_date"])
sales.head()


In [ ]:
end = sales["sold_date"].max()
start = end - pd.Timedelta(weeks=EVAL_WEEKS) + pd.Timedelta(days=1)
result = rolling_forecast(sales, start, end)

print(f"평가 기간 {start.date()} ~ {end.date()} · {len(result)}건 (날짜 × 채널 × 메뉴)")
print(f"WAPE {wape(result.actual, result.predicted):.3f} · Bias {bias(result.actual, result.predicted):+.3f}")


## 채널별

배달과 홀을 따로 채점한다. 한쪽만 크게 틀리면 그 채널의 요일 패턴을 못 잡는다는 뜻이다.


In [ ]:
by_channel = result.groupby("channel").apply(score, include_groups=False)
by_channel.round(3)


## 채널 합산 — 재료 소비로 이어지는 값

발주에 쓰는 건 배달 + 홀을 더한 메뉴별 판매량이다. 채널별 예측을 더한 값으로 채점한다.


In [ ]:
total = result.groupby(["sold_date", "menu_name"], as_index=False)[["actual", "predicted"]].sum()
print(f"합산 WAPE {wape(total.actual, total.predicted):.3f} · Bias {bias(total.actual, total.predicted):+.3f}")


## 메뉴별


In [ ]:
by_menu = result.groupby(["menu_name", "channel"]).apply(score, include_groups=False)
by_menu["WAPE"].unstack("channel").round(3)


## 요일별

금 · 토(배달은 일요일까지)가 다른 요일보다 크게 틀리면 요일 패턴을 못 잡는다는 뜻이다.


In [ ]:
DOW = ["월", "화", "수", "목", "금", "토", "일"]
result["요일"] = result["sold_date"].dt.dayofweek.map(dict(enumerate(DOW)))
by_dow = result.groupby(["요일", "channel"]).apply(lambda g: wape(g.actual, g.predicted), include_groups=False)
by_dow.unstack("channel").reindex(DOW).round(3)
